<a href="https://colab.research.google.com/github/yehia2811ahmed-ship-it/Data-Cleaning-Projects/blob/main/Apache%20Log%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import files
import pandas as pd

# رفع الملف
uploaded = files.upload()


Saving Apache_2k.log to Apache_2k.log


In [8]:
import re
import pandas as pd

file_path = 'Apache_2k.log'

# New pattern based on your image: [Date/Time] [Type] Message
# Example: [Sun Dec 04 04:47:44 2005] [notice] workerEnv.init()...
log_pattern = r'\[(.*?)\] \[(.*?)\] (.*)'

data = []

with open(file_path, 'r') as f:
    for line in f:
        match = re.search(log_pattern, line)
        if match:
            data.append(match.groups())

# Creating the table with correct columns
columns = ['Full_Timestamp', 'Log_Level', 'Message']
df = pd.DataFrame(data, columns=columns)

# Show the result
df.head()


,Full_Timestamp,Log_Level,Message
0,Sun Dec 04 04:47:44 2005,notice,workerEnv.init() ok /etc/httpd/conf/workers2.p...
1,Sun Dec 04 04:47:44 2005,error,mod_jk child workerEnv in error state 6
2,Sun Dec 04 04:51:08 2005,notice,jk2_init() Found child 6725 in scoreboard slot 10
3,Sun Dec 04 04:51:09 2005,notice,jk2_init() Found child 6726 in scoreboard slot 8
4,Sun Dec 04 04:51:09 2005,notice,jk2_init() Found child 6728 in scoreboard slot 6


In [9]:
# Convert Full_Timestamp to actual datetime objects
# The format matches: Day Name, Month, Day Number, Time, Year
df['Full_Timestamp'] = pd.to_datetime(df['Full_Timestamp'], format='%a %b %d %H:%M:%S %Y')

# Let's also clean the Log_Level and Message columns from any extra spaces
df['Log_Level'] = df['Log_Level'].str.strip()
df['Message'] = df['Message'].str.strip()

# Show the data types to confirm the change
print(df.dtypes)
df.head()


Full_Timestamp    datetime64[ns]
Log_Level                 object
Message                   object
dtype: object


,Full_Timestamp,Log_Level,Message
0,2005-12-04 04:47:44,notice,workerEnv.init() ok /etc/httpd/conf/workers2.p...
1,2005-12-04 04:47:44,error,mod_jk child workerEnv in error state 6
2,2005-12-04 04:51:08,notice,jk2_init() Found child 6725 in scoreboard slot 10
3,2005-12-04 04:51:09,notice,jk2_init() Found child 6726 in scoreboard slot 8
4,2005-12-04 04:51:09,notice,jk2_init() Found child 6728 in scoreboard slot 6


In [10]:
# Split the Message column into 'Module' and 'Clean_Message'
# We will split at the first space found
df[['Module', 'Clean_Message']] = df['Message'].str.split(' ', n=1, expand=True)

# Now we can drop the original 'Message' column to keep the notebook clean
df = df.drop(columns=['Message'])

# Reorder columns for better readability
df = df[['Full_Timestamp', 'Log_Level', 'Module', 'Clean_Message']]

# Display the first 10 rows this time to see the variety
df.head(10)


,Full_Timestamp,Log_Level,Module,Clean_Message
0,2005-12-04 04:47:44,notice,workerEnv.init(),ok /etc/httpd/conf/workers2.properties
1,2005-12-04 04:47:44,error,mod_jk,child workerEnv in error state 6
2,2005-12-04 04:51:08,notice,jk2_init(),Found child 6725 in scoreboard slot 10
3,2005-12-04 04:51:09,notice,jk2_init(),Found child 6726 in scoreboard slot 8
4,2005-12-04 04:51:09,notice,jk2_init(),Found child 6728 in scoreboard slot 6
5,2005-12-04 04:51:14,notice,workerEnv.init(),ok /etc/httpd/conf/workers2.properties
6,2005-12-04 04:51:14,notice,workerEnv.init(),ok /etc/httpd/conf/workers2.properties
7,2005-12-04 04:51:14,notice,workerEnv.init(),ok /etc/httpd/conf/workers2.properties
8,2005-12-04 04:51:18,error,mod_jk,child workerEnv in error state 6
9,2005-12-04 04:51:18,error,mod_jk,child workerEnv in error state 6


In [11]:
# Check for any missing values (Nulls)
print("Missing values in each column:")
print(df.isnull().sum())

# Check for duplicate rows
duplicates_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates_count}")

# If you want to remove duplicates, uncomment the next line:
# df = df.drop_duplicates()


Missing values in each column:
Full_Timestamp    0
Log_Level         0
Module            0
Clean_Message     0
dtype: int64

Number of duplicate rows: 539


In [12]:
# 1. Remove duplicate rows
df = df.drop_duplicates()
print(f"Duplicates removed. New total rows: {len(df)}")

Duplicates removed. New total rows: 1461


In [13]:
# Check for any missing values (Nulls)
print("Missing values in each column:")
print(df.isnull().sum())

# Check for duplicate rows
duplicates_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates_count}")

# If you want to remove duplicates, uncomment the next line:
# df = df.drop_duplicates()


Missing values in each column:
Full_Timestamp    0
Log_Level         0
Module            0
Clean_Message     0
dtype: int64

Number of duplicate rows: 0


In [15]:
!pip install xlsxwriter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.4 MB/s eta 0:00:00


In [16]:
# 1. Prepare the filename
output_file = 'Cleaned_Apache_Logs.xlsx'

# 2. Use ExcelWriter with xlsxwriter engine for professional formatting
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='Logs_Report', index=False)

    workbook  = writer.book
    worksheet = writer.sheets['Logs_Report']

    # Add a header format (Bold, background color, border)
    header_format = workbook.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'vcenter',
        'fg_color': '#D7E4BC',
        'border': 1
    })

    # Apply formatting to the headers and set column widths
    for col_num, value in enumerate(df.columns.values):
        worksheet.write(0, col_num, value, header_format)
        # Set column width to 25 to make it look clean
        worksheet.set_column(col_num, col_num, 25)

    # Add a filter to the columns
    worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)

# 3. Download the file to your computer
from google.colab import files
files.download(output_file)

print("Done! Your professional Excel file is ready and downloading...")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Your professional Excel file is ready and downloading...


# 📊 Apache Log Data Cleaning & Transformation Report

## 📝 Project Overview
This project aims to transform raw, unstructured **Apache Error Logs** into a clean, structured, and professional **Excel report**. The process involved parsing complex text patterns, cleaning data, and enriching it for better analysis.

---

## 🛠️ Tools & Technologies Used
*   **Python**: The core programming language.
*   **Pandas**: For data manipulation and tabular structure (DataFrames).
*   **Regex (re)**: To parse and extract specific information from raw log lines.
*   **XlsxWriter**: To create a professionally formatted Excel file with filters and styling.

---

## ⚙️ Steps Taken (Data Pipeline)

### 1. Data Ingestion
*   Loaded the raw `Apache_2k.log` file into the environment.

### 2. Regex Parsing (Structuring)
*   Used **Regular Expressions** to break down each text line into three primary components:
    *   `Full_Timestamp`: The exact date and time of the log.
    *   `Log_Level`: The severity (e.g., *notice*, *error*).
    *   `Message`: The full descriptive text.

### 3. Data Cleaning & Type Conversion
*   **Datetime Normalization**: Converted the text-based timestamp into a standard Python `datetime` object for time-series analysis.
*   **String Trimming**: Removed extra spaces and hidden characters from all columns.
*   **Deduplication**: Identified and removed **539 duplicate entries** to ensure data integrity and avoid skewed results.

### 4. Feature Extraction (Advanced Splitting)
*   Extracted the **Module name** (e.g., `workerEnv.init`, `mod_jk`) from the main message to allow for more granular analysis of which system components are reporting issues.

### 5. Professional Export
*   Generated an **Excel file** with:
    *   **Auto-filters** enabled for easy searching.
    *   **Custom formatting** (Header colors, bold text).
    *   **Optimized column widths** for immediate readability.

---
**Done by: [Yehia]**
